# Kaggle Training: LogFilter CrossEncoder on log ↔ ATT&CK pairs

Fine-tunes `cisco-ai/SecureBERT2.0-cross_encoder` to score `(log_text, ATT&CK technique description)` pairs. The published variant was trained on CTI-prose pairs, not log-vs-CTI pairs. In `LogScorer` the CrossEncoder weight is `0.35` — the largest single contribution to `ai_threat_score` (see [config/config.yaml](../config/config.yaml)) — so its calibration on log-domain text directly drives composite-score quality.

Bootstrap signal comes from a small keyword→technique map applied to the HDFS TraceBench reconstructed corpus. This produces high-precision positive pairs (when a log line uses a keyword that's diagnostic of an ATT&CK technique, that pair is a positive) and lets us draw negatives at random from non-matching techniques. The result is a binary-classification CrossEncoder fit on log-shaped left-hand sides and analyst-prose right-hand sides — exactly the production input distribution.

Output: `models/cross_encoder/final/` — a sentence-transformers CrossEncoder directory drop-in for [src/logfilter/models/cross_encoder.py](../src/logfilter/models/cross_encoder.py). Expected runtime on Kaggle Free GPU (T4) is well under the 9h session limit. Enable a GPU accelerator before running training cells.

## 1. Locate the repo

Run this notebook from the project root when possible. If Kaggle places the repo inside `/kaggle/working`, the cell below tries to find it.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

KAGGLE_WORKING = Path('/kaggle/working')
REPO_DIR = Path.cwd()
GIT_URL = 'https://github.com/Jacob-Valor/Modern-AI-Log-Filter-Training.git'

if not (REPO_DIR / 'training' / 'train.py').exists():
    matches = list(KAGGLE_WORKING.glob('*/training/train.py'))
    if matches:
        REPO_DIR = matches[0].parents[1]
    else:
        clone_dir = KAGGLE_WORKING / 'Modern-AI-Log-Filter-Training'
        if not clone_dir.exists():
            subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, str(clone_dir)], check=True)
        REPO_DIR = clone_dir

if not (REPO_DIR / 'training' / 'train.py').exists():
    raise FileNotFoundError('Could not find training/train.py. Upload or clone the repo into /kaggle/working first.')

os.chdir(REPO_DIR)
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('TORCHDYNAMO_DISABLE', '1')

sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo:', REPO_DIR)

## 2. Install training dependencies

Adds `sentence-transformers` (which provides the `CrossEncoder` training loop) on top of the standard transformers/datasets stack.

In [ ]:
%pip install -q transformers datasets accelerate sentence-transformers scikit-learn

## 3. Attach the HDFS TraceBench dataset and load MITRE techniques

The training data is built from two existing repo artifacts: the reconstructed text windows (left-hand side of each pair) and `config/mitre_techniques.json` (right-hand side — technique name + description, the same input the production CrossEncoder receives at inference).

In [ ]:
PREPROCESSED_DIR = REPO_DIR / 'HDFS_v3_TraceBench' / 'preprocessed'
MITRE_PATH = REPO_DIR / 'config' / 'mitre_techniques.json'

def has_trace_files(path: Path) -> bool:
    return (path / 'normal_trace.csv').exists() and (path / 'failure_trace.csv').exists()

if not has_trace_files(PREPROCESSED_DIR):
    candidates: list[Path] = []
    search_log: list[str] = []

    # Strategy 1: try a small set of known mount paths (no filesystem walk, very fast)
    KNOWN_PATHS = [
        Path('/kaggle/input/datasets/jacobvalor/hdfs-tracebench-preprocessed-logs'),
        Path('/kaggle/input/datasets/jacobvalor/hdfs-tracebench-preprocessed-logs/preprocessed'),
        Path('/kaggle/input/jacobvalor-hdfs-tracebench-preprocessed-logs'),
        Path('/kaggle/input/jacobvalor-hdfs-tracebench-preprocessed-logs/preprocessed'),
        Path('/kaggle/input/hdfs-tracebench-preprocessed-logs'),
        Path('/kaggle/input/hdfs-tracebench-preprocessed-logs/preprocessed'),
    ]
    for p in KNOWN_PATHS:
        if has_trace_files(p):
            candidates.append(p)
            search_log.append(f'known-path hit: {p}')
            break
        search_log.append(f'known-path miss: {p} (exists={p.exists()})')

    # Strategy 2: bounded top-level search of /kaggle/input/ (one level, not recursive)
    if not candidates:
        input_root = Path('/kaggle/input')
        if input_root.exists():
            try:
                for d in input_root.iterdir():
                    if d.is_dir() and has_trace_files(d):
                        candidates.append(d)
                        search_log.append(f'top-level dir hit: {d}')
                        break
                    if d.is_dir():
                        search_log.append(f'top-level dir no-trace-files: {d}')
                    else:
                        search_log.append(f'top-level non-dir: {d}')
            except Exception as e:
                search_log.append(f'iterdir() failed: {e}')
        else:
            search_log.append('/kaggle/input/ does NOT exist')

    if not candidates:
        diag = ['(no /kaggle/input/ contents available)']
        if Path('/kaggle/input').exists():
            try:
                diag = []
                for d in Path('/kaggle/input').iterdir():
                    if d.is_dir():
                        try:
                            files = sorted(p.name for p in d.iterdir())[:10]
                        except Exception as e:
                            files = [f'<iterdir failed: {e}>']
                        diag.append(f'/kaggle/input/{d.name}/ -> {files}')
                    else:
                        diag.append(f'/kaggle/input/{d.name} (not a dir)')
            except Exception as e:
                diag = [f'iterdir() failed: {e}']
        raise FileNotFoundError(
            f'No HDFS trace files found.\n'
            f'Search log:\n  ' + '\n  '.join(search_log) + '\n'
            f'/kaggle/input/ contents:\n  ' + '\n  '.join(diag)
        )

    source = candidates[0]
    PREPROCESSED_DIR.parent.mkdir(parents=True, exist_ok=True)
    if PREPROCESSED_DIR.exists() and not PREPROCESSED_DIR.is_symlink():
        raise FileExistsError(f'{PREPROCESSED_DIR} exists but does not contain the expected trace files.')
    if not PREPROCESSED_DIR.exists():
        os.symlink(source, PREPROCESSED_DIR, target_is_directory=True)
    print('Linked dataset:', source, '->', PREPROCESSED_DIR)
    print('Search log:\n  ' + '\n  '.join(search_log))
else:
    print('Dataset available:', PREPROCESSED_DIR)

if not MITRE_PATH.exists():
    raise FileNotFoundError(f'Missing MITRE technique seed: {MITRE_PATH}')
techniques = json.loads(MITRE_PATH.read_text())
tech_by_id = {t['id']: t for t in techniques}
print(f'Loaded {len(techniques)} ATT&CK techniques')


## 4. Build positive / negative pairs from keyword heuristics

`KEYWORD_MAP` is the single hand-crafted asset in this notebook — each entry maps a log-surface keyword to the ATT&CK technique IDs it is diagnostic of. Coverage is intentionally narrow (~20 entries) because precision matters more than recall for bootstrap labels: a noisy positive pair is far worse than a missing one. Extend the map for new log domains rather than relaxing the matching rule.

For each text window:
* Every keyword hit produces a positive `(text, technique_text, 1.0)` pair against each mapped technique.
* For balance, we draw `NEG_PER_POS` random non-matching techniques as negative `(text, technique_text, 0.0)` pairs.

Texts with zero keyword hits are dropped from training (no signal). The final dataset is shuffled and split 90/10 train/eval.

In [ ]:
import random
from training.text_dataset import build_windows

# Keyword → ATT&CK technique IDs. Keep keys lowercase; matching is case-insensitive.
# Extend conservatively when adding new log sources; high precision > high recall here.
KEYWORD_MAP: dict[str, list[str]] = {
    # Auth / credentials
    'permission denied': ['T1078'],
    'access denied': ['T1078'],
    'accesscontrolexception': ['T1078'],
    'authentication failed': ['T1110'],
    'authorization failed': ['T1078'],
    'invalid credentials': ['T1110'],
    'login failed': ['T1110'],
    'kerberos': ['T1003'],
    'credential': ['T1003'],
    # Remote services / lateral
    'ssh': ['T1021', 'T1021.004'],
    'rdp': ['T1021.001'],
    'smb://': ['T1021.002'],
    '\\\\': ['T1021.002'],          # UNC path
    'winrm': ['T1021.006'],
    'telnet': ['T1021'],
    # Execution / scripting
    'powershell': ['T1059.001'],
    'cmd.exe': ['T1059.003'],
    '/bin/sh': ['T1059.004'],
    '/bin/bash': ['T1059.004'],
    'wscript': ['T1059.005'],
    # Network / C2
    'connection reset': ['T1071'],
    'connection refused': ['T1071'],
    # DoS / impact
    'outofmemory': ['T1499'],
    'java.lang.outofmemoryerror': ['T1499'],
    # HDFS-Hadoop specific (Remote Services proxy)
    'dataxceiver': ['T1021'],
    'write_block': ['T1021'],
    'read_block': ['T1021'],
}

NEG_PER_POS = 2
RANDOM_SEED = 42
MAX_TEXT_CHARS = 1500   # truncate left side aggressively; CE pair fits in 1024 tokens

def technique_text(tid: str) -> str | None:
    t = tech_by_id.get(tid)
    if t is None:
        return None
    return f"{t['name']}. {t['description']}"

def matched_techniques(text: str) -> set[str]:
    lowered = text.lower()
    out: set[str] = set()
    for kw, tids in KEYWORD_MAP.items():
        if kw in lowered:
            for tid in tids:
                if tid in tech_by_id:
                    out.add(tid)
    return out

def build_pairs(
    sample_normal: int | None,
    sample_failure: int | None,
    seed: int = RANDOM_SEED,
) -> list[tuple[str, str, float]]:
    texts, _, _ = build_windows(
        sample_normal=sample_normal,
        sample_failure=sample_failure,
        random_state=seed,
    )
    rng = random.Random(seed)
    all_tids = list(tech_by_id.keys())
    pairs: list[tuple[str, str, float]] = []
    for text in texts:
        text_trim = text[:MAX_TEXT_CHARS]
        matched = matched_techniques(text)
        if not matched:
            continue
        for tid in matched:
            tt = technique_text(tid)
            if tt:
                pairs.append((text_trim, tt, 1.0))
        # Negatives — sample random non-matching techniques
        non_matching = [t for t in all_tids if t not in matched]
        n_neg = min(NEG_PER_POS * len(matched), len(non_matching))
        if n_neg <= 0:
            continue
        for tid in rng.sample(non_matching, n_neg):
            tt = technique_text(tid)
            if tt:
                pairs.append((text_trim, tt, 0.0))
    rng.shuffle(pairs)
    return pairs

# Sanity sample
sample_pairs = build_pairs(sample_normal=2000, sample_failure=2000, seed=42)
n_pos = sum(1 for _, _, lbl in sample_pairs if lbl == 1.0)
n_neg = sum(1 for _, _, lbl in sample_pairs if lbl == 0.0)
print(f'sampled pairs: {len(sample_pairs)}  ({n_pos} positives / {n_neg} negatives)')
if sample_pairs:
    log_t, tech_t, lbl = sample_pairs[0]
    print('\n--- example pair ---')
    print(f'label: {lbl}')
    print(f'log: {log_t[:300]}{"..." if len(log_t) > 300 else ""}')
    print(f'technique: {tech_t[:300]}{"..." if len(tech_t) > 300 else ""}')

## 5. Sampled CrossEncoder run (verify environment first)

Train two epochs on the 2K-normal + 2K-failure-derived pair set with binary classification (sigmoid head, BCE loss). The sentence-transformers `CrossEncoder.fit()` handles tokenisation and the training loop. Output: `models/cross_encoder-sample/final/`.

In [ ]:
import math
import torch
from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
from torch.utils.data import DataLoader

# To consume the MLM-adapted base produced by kaggle_pretrain_mlm.ipynb,
# replace MODEL_ID with 'models/securebert2-logs-mlm/final' (the CrossEncoder
# wrapper will reinitialise the scoring head on first .fit()).
MODEL_ID = 'cisco-ai/SecureBERT2.0-cross_encoder'
MAX_LENGTH = 1024
EPOCHS = 2
BATCH_SIZE = 8
LR = 2e-5

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16
print('CUDA:', torch.cuda.is_available(), 'bf16:', use_bf16, 'fp16:', use_fp16)

split = int(0.9 * len(sample_pairs))
train_pairs, eval_pairs = sample_pairs[:split], sample_pairs[split:]
train_examples = [InputExample(texts=[lt, tt], label=lbl) for lt, tt, lbl in train_pairs]
eval_examples = [(lt, tt, lbl) for lt, tt, lbl in eval_pairs]
print(f'train={len(train_examples)}  eval={len(eval_examples)}')

model = CrossEncoder(MODEL_ID, num_labels=1, max_length=MAX_LENGTH)

train_loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
eval_sentence_pairs = [[lt, tt] for lt, tt, _ in eval_examples]
eval_labels = [int(lbl) for _, _, lbl in eval_examples]
evaluator = CEBinaryClassificationEvaluator(
    sentence_pairs=eval_sentence_pairs,
    labels=eval_labels,
    name='ce-logs-attack-sample',
)

warmup_steps = math.ceil(len(train_loader) * EPOCHS * 0.1)
sample_out = REPO_DIR / 'models' / 'cross_encoder-sample' / 'final'
sample_out.mkdir(parents=True, exist_ok=True)

model.fit(
    train_dataloader=train_loader,
    evaluator=evaluator,
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={'lr': LR},
    use_amp=(use_bf16 or use_fp16),
    show_progress_bar=True,
    output_path=str(sample_out),
    save_best_model=True,
)
model.save(str(sample_out), create_model_card=False)

# Standalone metrics (sentence-transformers writes a CSV; we also dump JSON)
from sklearn.metrics import roc_auc_score, average_precision_score
scores = model.predict(eval_sentence_pairs, batch_size=BATCH_SIZE, apply_softmax=False)
metrics = {
    'n_eval': len(eval_labels),
    'pos_rate': float(sum(eval_labels) / max(len(eval_labels), 1)),
    'roc_auc': float(roc_auc_score(eval_labels, scores)) if len(set(eval_labels)) > 1 else None,
    'avg_precision': float(average_precision_score(eval_labels, scores)) if len(set(eval_labels)) > 1 else None,
}
(sample_out / 'cross_encoder_metrics.json').write_text(json.dumps(metrics, indent=2))
print('sampled eval:', metrics)
print('saved:', sample_out)

## 6. Full CrossEncoder run (uncomment when sampled run succeeds)

Use the entire HDFS reconstructed corpus to generate pairs. Most normal traces will produce zero positives (no keyword hits) and be skipped; the data volume comes from failure traces where exception text is keyword-rich. Output: `models/cross_encoder/final/`. Estimated runtime ~1-3h on Kaggle T4.

In [ ]:
full_pairs = build_pairs(sample_normal=None, sample_failure=None, seed=42)
n_pos_f = sum(1 for _, _, lbl in full_pairs if lbl == 1.0)
n_neg_f = sum(1 for _, _, lbl in full_pairs if lbl == 0.0)
print(f'full pairs: {len(full_pairs)}  ({n_pos_f} positives / {n_neg_f} negatives)')

split = int(0.9 * len(full_pairs))
train_pairs_f, eval_pairs_f = full_pairs[:split], full_pairs[split:]
train_examples_f = [InputExample(texts=[lt, tt], label=lbl) for lt, tt, lbl in train_pairs_f]
eval_sentence_pairs_f = [[lt, tt] for lt, tt, _ in eval_pairs_f]
eval_labels_f = [int(lbl) for _, _, lbl in eval_pairs_f]

train_loader_f = DataLoader(train_examples_f, shuffle=True, batch_size=BATCH_SIZE)
evaluator_f = CEBinaryClassificationEvaluator(
    sentence_pairs=eval_sentence_pairs_f, labels=eval_labels_f,
    name='ce-logs-attack-full',
)
warmup_f = math.ceil(len(train_loader_f) * EPOCHS * 0.1)
full_out = REPO_DIR / 'models' / 'cross_encoder' / 'final'
full_out.mkdir(parents=True, exist_ok=True)

model_f = CrossEncoder(MODEL_ID, num_labels=1, max_length=MAX_LENGTH)
model_f.fit(
    train_dataloader=train_loader_f,
    evaluator=evaluator_f,
    epochs=EPOCHS,
    warmup_steps=warmup_f,
    optimizer_params={'lr': LR},
    use_amp=(use_bf16 or use_fp16),
    show_progress_bar=True,
    output_path=str(full_out),
    save_best_model=True,
)
model_f.save(str(full_out), create_model_card=False)
scores_f = model_f.predict(eval_sentence_pairs_f, batch_size=BATCH_SIZE, apply_softmax=False)
metrics_f = {
    'n_eval': len(eval_labels_f),
    'pos_rate': float(sum(eval_labels_f) / max(len(eval_labels_f), 1)),
    'roc_auc': float(roc_auc_score(eval_labels_f, scores_f)) if len(set(eval_labels_f)) > 1 else None,
    'avg_precision': float(average_precision_score(eval_labels_f, scores_f)) if len(set(eval_labels_f)) > 1 else None,
}
(full_out / 'cross_encoder_metrics.json').write_text(json.dumps(metrics_f, indent=2))
print('full eval:', metrics_f)


## 7. Inspect artifacts

Confirm the output directory has the standard sentence-transformers CrossEncoder layout.

In [ ]:
for candidate in [
    REPO_DIR / 'models' / 'cross_encoder' / 'final',
    REPO_DIR / 'models' / 'cross_encoder-sample' / 'final',
]:
    if not candidate.exists():
        continue
    print(f'=== {candidate} ===')
    for path in sorted(candidate.rglob('*')):
        if path.is_file():
            print(f'  {path.relative_to(candidate)}: {path.stat().st_size:,} bytes')
    metrics_path = candidate / 'cross_encoder_metrics.json'
    if metrics_path.exists():
        print('  metrics:', json.loads(metrics_path.read_text()))

## 8. Package artifacts

Zip the final model directory into `/kaggle/working/`.

In [ ]:
import shutil

tag = 'full' if (REPO_DIR / 'models' / 'cross_encoder' / 'final').exists() else 'sample'
src_dir = REPO_DIR / ('models/cross_encoder/final' if tag == 'full' else 'models/cross_encoder-sample/final')
if not src_dir.exists():
    raise FileNotFoundError('No CrossEncoder artifact found. Run the sampled or full training cell first.')

out_zip = KAGGLE_WORKING / f'logfilter-cross-encoder-{tag}-artifacts'
shutil.make_archive(
    str(out_zip), 'zip',
    root_dir=REPO_DIR,
    base_dir=Path(*src_dir.parts[-3:]).as_posix(),
)
print('packaged:', out_zip.with_suffix('.zip'))

## 9. Output description + how to consume in repo

Download `/kaggle/working/logfilter-cross-encoder-full-artifacts.zip` (or `-sample-` for the verification run) and extract it at the repository root. The archive contains:

```text
models/cross_encoder/final/config.json
models/cross_encoder/final/model.safetensors
models/cross_encoder/final/tokenizer.json (and tokenizer_config.json, special_tokens_map.json)
models/cross_encoder/final/cross_encoder_metrics.json
```

To use this fine-tuned CrossEncoder inside the production pipeline, point [src/logfilter/models/cross_encoder.py](../src/logfilter/models/cross_encoder.py) at the local directory:

```yaml
# config/config.yaml
models:
  cross_encoder:
    model_id: "models/cross_encoder/final"   # was: cisco-ai/SecureBERT2.0-cross_encoder
    device: "cpu"
    batch_size: 16
```

Caveats:

* **Bootstrap labels are heuristic.** ROC-AUC and average precision in `cross_encoder_metrics.json` measure agreement with the keyword map, not absolute correctness against an analyst-curated benchmark. Treat them as a *relative* signal between training runs and as a regression check.
* **Score scale changes.** Fine-tuning shifts the CrossEncoder's score distribution — your existing `cross_encoder=0.35` weight in [config/config.yaml](../config/config.yaml) was tuned against the published variant's distribution. Re-calibrate the composite score (PR curve over `ai_threat_score`) after deploying this artifact, and adjust the `routing.high/medium/low` thresholds before relying on the new model in `suppress_low` mode.
* **Composability with MLM pre-training.** For best results, run [kaggle_pretrain_mlm.ipynb](kaggle_pretrain_mlm.ipynb) first and switch `MODEL_ID` here to `models/securebert2-logs-mlm/final` so the encoder is log-adapted before the cross-encoder head is trained.
* **Coverage extension.** Add domains (auth logs, web/app server, Windows Security) by extending `KEYWORD_MAP` rather than weakening the matching rule. Each new key should be a high-precision indicator of one or two ATT&CK techniques.